# NERISAL: DM-Count density model → ExecuTorch (.pte), Google Colab

DM-Count (NeurIPS 2020, MIT licence) estimates a **density map**: every person adds up to 1, so the sum is the count.
We use it only for packed crowds, where heads are too small to box.

This notebook:
1. downloads the official code and pretrained weights (ShanghaiTech B = street crowds, QNRF = very dense crowds)
2. checks the counts in PyTorch at full resolution and at our Pi sizes
3. exports to `.pte` at **256×144** and **432×240** and verifies each file
4. zips everything for the Builder to time on the Pi

Runtime: **CPU is fine** (Runtime → Change runtime type → CPU). About 15–20 minutes.

In [ ]:
# 1. Install (ExecuTorch pinned to the Pi's version) and get the official DM-Count code + weights
!pip install -q executorch==1.6.0 gdown
!pip show executorch | grep Version
!git clone -q https://github.com/cvlab-stonybrook/DM-Count.git
!gdown -q --folder https://drive.google.com/drive/folders/10U7F4iW_aPICM5-qJq21SXLLkzlum9tX -O dm_weights
import glob, os
W = sorted(glob.glob("dm_weights/**/*.pth", recursive=True))
print("weights:", [os.path.basename(w) + f" ({os.path.getsize(w)/1e6:.0f} MB)" for w in W])

In [ ]:
# 2. Load a model (builds VGG-19 + DM-Count head, then loads the pretrained weights)
import sys, torch, numpy as np
sys.path.insert(0, "DM-Count")
from models import vgg19

def load(name):
    path = [w for w in W if name in os.path.basename(w).lower()]
    assert path, f"no weights matching '{name}' in {W}"
    m = vgg19()                                         # downloads ImageNet VGG-19 once (backbone shape)
    m.load_state_dict(torch.load(path[0], map_location="cpu"))
    return m.eval()

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
def prep(img_rgb, size=None):
    """img_rgb: HxWx3 uint8. size: (w, h) to resize to, or None for full resolution."""
    import cv2
    if size: img_rgb = cv2.resize(img_rgb, size, interpolation=cv2.INTER_AREA)
    x = (img_rgb.astype(np.float32) / 255 - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)

models = {}
for name in ("shb", "qnrf"):
    try:
        models[name] = load(name); print("loaded", name)
    except AssertionError as e:
        print(e)

In [ ]:
# 3. Test images: the repo's examples + any crowd photos you upload (drag into the file panel, folder 'test_imgs')
import cv2
os.makedirs("test_imgs", exist_ok=True)
imgs = sorted(glob.glob("DM-Count/example_images/*") + glob.glob("test_imgs/*"))
print(len(imgs), "test images:", [os.path.basename(i) for i in imgs])

In [ ]:
# 4. Count check: full resolution vs our Pi input sizes (shows how much shrinking the image costs)
SIZES = {"full": None, "432x240": (432, 240), "256x144": (256, 144)}
rows = []
for p in imgs:
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    for name, m in models.items():
        r = [os.path.basename(p)[:20], name]
        for s in SIZES.values():
            with torch.no_grad():
                mu, _ = m(prep(im, s))
            r.append(f"{mu.sum().item():.0f}")
        rows.append(r)
print(f"{'image':22s}{'model':6s}" + "".join(f"{k:>10s}" for k in SIZES))
for r in rows:
    print(f"{r[0]:22s}{r[1]:6s}" + "".join(f"{v:>10s}" for v in r[2:]))

In [ ]:
# 5. Export to ExecuTorch (.pte, XNNPACK) at both Pi sizes; the exported model returns the density map only
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

class DensityOnly(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        return self.m(x)[0]          # density map (1, 1, H/8, W/8); sum = count

os.makedirs("pte", exist_ok=True)
for name, m in models.items():
    for w, h in ((432, 240), (256, 144)):
        x = torch.randn(1, 3, h, w)
        with torch.no_grad():
            ep = torch.export.export(DensityOnly(m), (x,))
        prog = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()]).to_executorch()
        out = f"pte/dmcount_{name}_{w}x{h}.pte"
        open(out, "wb").write(prog.buffer)
        print(out, f"{os.path.getsize(out)/1e6:.0f} MB")

In [ ]:
# 6. Verify each .pte: runs, right output shape, and gives the same count as PyTorch
from executorch.runtime import Runtime
import time
test = cv2.cvtColor(cv2.imread(imgs[0]), cv2.COLOR_BGR2RGB)
for f in sorted(glob.glob("pte/*.pte")):
    name = f.split("_")[1]; w, h = map(int, f.split("_")[-1][:-4].split("x"))
    x = prep(test, (w, h))
    meth = Runtime.get().load_program(f).load_method("forward")
    t = time.perf_counter(); d = meth.execute([x])[0]; ms = (time.perf_counter() - t) * 1000
    with torch.no_grad():
        ref = models[name](x)[0].sum().item()
    print(f"{os.path.basename(f):28s} out {tuple(d.shape)}  count .pte {d.sum().item():7.1f}  pytorch {ref:7.1f}  "
          f"({ms:.0f} ms on Colab CPU, Pi will differ)")

In [ ]:
# 7. Zip for the Builder (then B times them on the Pi: decision gate is < 2 s per frame)
import zipfile
with zipfile.ZipFile("nerisal_dmcount_pte.zip", "w") as z:
    for f in glob.glob("pte/*.pte"):
        z.write(f, os.path.basename(f))
print(zipfile.ZipFile("nerisal_dmcount_pte.zip").namelist())
from google.colab import files; files.download("nerisal_dmcount_pte.zip")